# ECE 447 — Project B: Anomaly Detection (final submission)

Integrated notebook: **Member A** (data, framing, statistical method) and **Member C** (Isolation Forest, metrics, artifacts). **Member B** (distance-based) is a clearly marked placeholder so the notebook still runs end-to-end.

**Paths:** `PROJECT_ROOT = Path("../..").resolve()` when this file lives in `notebooks/final/`.


## 1) Problem framing (Member A + team)

- **Anomaly:** Credit card transactions labeled fraudulent (`Class == 1`) vs genuine (`0`).
- **Learning setting:** **Semi-supervised** — unsupervised/scoring models use **normal-only** training rows; val/test include fraud for threshold tuning and evaluation (`data_pipeline` matches Member A).
- **Constraints:** Extreme **imbalance** (~0.4% fraud); report **precision/recall/F1/FPR**, not accuracy alone; `V1`–`V28` are anonymized PCA features.
- **Data:** ULB Kaggle Credit Card Fraud; loading via [`src/data_pipeline.py`](../../src/data_pipeline.py).


In [ ]:
from __future__ import annotations

import importlib.util
import sys
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Image, display as idisplay
from sklearn.metrics import ConfusionMatrixDisplay, PrecisionRecallDisplay, RocCurveDisplay

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

PROJECT_ROOT = Path("../..").resolve()
SRC = PROJECT_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

ARTIFACT_DIR = PROJECT_ROOT / "artifacts"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)


def _import_src_module(name: str, path: Path):
    spec = importlib.util.spec_from_file_location(name, path)
    if spec is None or spec.loader is None:
        raise ImportError(f"Cannot load {path}")
    mod = importlib.util.module_from_spec(spec)
    sys.modules[name] = mod
    spec.loader.exec_module(mod)
    return mod


_import_src_module("data_pipeline", SRC / "data_pipeline.py")
_import_src_module("ml_detector", SRC / "ml_detector.py")

from data_pipeline import RANDOM_STATE as PIPE_SEED, load_credit_card_splits
from ml_detector import (
    fit_isolation_forest,
    metrics_at_threshold,
    predict_with_threshold,
    sweep_thresholds_val,
)

assert PIPE_SEED == RANDOM_STATE
print("PROJECT_ROOT:", PROJECT_ROOT)


## 2) Data loading (Member A pipeline)

Shared splits: `artifacts/*.npy` if present, else CSV / Kaggle resolve.


In [ ]:
X_train, X_val, X_test, y_train, y_val, y_test = load_credit_card_splits(PROJECT_ROOT)

FEATURE_NAMES = [f"V{i}" for i in range(1, 29)] + ["Time", "Amount"]
print("Train / val / test:", X_train.shape, X_val.shape, X_test.shape)
print("Fraud rate val:", f"{y_val.mean():.4%}", "| test:", f"{y_test.mean():.4%}")


## 3) Statistical anomaly detector (Member A slot)

**Implemented here:** robust per-feature z-scores (median + MAD on **training** normals); point score = **max** |robust z| across features. Threshold on **validation** via max **F1** over quantiles.

*Align or replace with Member A’s final exported method when their notebook is finalized.*


In [ ]:
def fit_robust_mad_params(X_ref: np.ndarray):
    med = np.median(X_ref, axis=0)
    mad = np.median(np.abs(X_ref - med), axis=0)
    mad = np.where(mad < 1e-12, 1e-12, mad)
    return med, mad


def statistical_anomaly_scores(X: np.ndarray, med: np.ndarray, mad: np.ndarray) -> np.ndarray:
    z = 0.6745 * (X - med) / mad
    return np.max(np.abs(z), axis=1)


med, mad = fit_robust_mad_params(X_train)
scores_stat_val = statistical_anomaly_scores(X_val, med, mad)
scores_stat_test = statistical_anomaly_scores(X_test, med, mad)


def pick_threshold_max_f1(scores: np.ndarray, y_true: np.ndarray, quantiles=None):
    if quantiles is None:
        quantiles = np.linspace(0.90, 0.9999, 50)
    best_f1, best_thr = -1.0, float(np.quantile(scores, 0.99))
    for q in quantiles:
        thr = float(np.quantile(scores, q))
        pred = (scores >= thr).astype(int)
        m = metrics_at_threshold(y_true, pred)
        if m["f1"] > best_f1:
            best_f1, best_thr = m["f1"], thr
    return best_thr


THRESHOLD_STAT = pick_threshold_max_f1(scores_stat_val, y_val)
pred_stat_val = (scores_stat_val >= THRESHOLD_STAT).astype(int)
pred_stat_test = (scores_stat_test >= THRESHOLD_STAT).astype(int)

print("Statistical — val:", metrics_at_threshold(y_val, pred_stat_val))
print("Statistical — test:", metrics_at_threshold(y_test, pred_stat_test))
print("Threshold (stat):", THRESHOLD_STAT)


## 4) Distance-based method (Member B) — placeholder

When Member B delivers artifacts, save e.g. `artifacts/memberB_scores.npz` with arrays **`scores_val`**, **`scores_test`** (same row order as `load_credit_card_splits`), and optionally **`threshold`**, then set **`MEMBER_B_READY = True`** in the next cell.


In [ ]:
MEMBER_B_READY = False
MEMBER_B_NPZ = ARTIFACT_DIR / "memberB_scores.npz"

scores_dist_val = scores_dist_test = None
thr_dist = None

if MEMBER_B_READY and MEMBER_B_NPZ.is_file():
    b = np.load(MEMBER_B_NPZ)
    scores_dist_val = np.asarray(b["scores_val"]).ravel()
    scores_dist_test = np.asarray(b["scores_test"]).ravel()
    thr_dist = float(b["threshold"]) if "threshold" in b.files else None
    print("Loaded Member B scores from", MEMBER_B_NPZ)
else:
    print("Member B: skipped (set MEMBER_B_READY and add npz when ready).")


## 5) Machine learning — Isolation Forest (Member C)

Prefer saved `iforest_memberC.joblib`; otherwise fit a default IF + validation sweep.


In [ ]:
MODEL_PATH = ARTIFACT_DIR / "iforest_memberC.joblib"

if MODEL_PATH.is_file():
    bundle = joblib.load(MODEL_PATH)
    if_model = bundle["model"]
    THRESHOLD_IF = float(bundle["threshold"])
    print("Loaded Isolation Forest from", MODEL_PATH)
else:
    print("No saved model — fitting IF + sweep (may take ~1–2 min).")
    if_model = fit_isolation_forest(
        X_train,
        n_estimators=400,
        max_samples=512,
        contamination=0.0005,
        random_state=RANDOM_STATE,
    )
    THRESHOLD_IF, _ = sweep_thresholds_val(if_model, X_val, y_val)

val_result = predict_with_threshold(if_model, X_val, THRESHOLD_IF)
test_result = predict_with_threshold(if_model, X_test, THRESHOLD_IF)

print("IF — val:", {k: metrics_at_threshold(y_val, val_result.pred_label)[k] for k in ("precision", "recall", "f1", "fpr")})
print("IF — test:", {k: metrics_at_threshold(y_test, test_result.pred_label)[k] for k in ("precision", "recall", "f1", "fpr")})


## 6) Thresholds and score distributions (test)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

ax = axes[0]
ax.hist(scores_stat_test[y_test == 0], bins=50, alpha=0.6, label="normal", density=True)
ax.hist(scores_stat_test[y_test == 1], bins=30, alpha=0.6, label="fraud", density=True)
ax.axvline(THRESHOLD_STAT, color="k", ls="--", label="threshold")
ax.set_title("Statistical score (test)")
ax.legend()

ax = axes[1]
s = test_result.anomaly_score
ax.hist(s[y_test == 0], bins=50, alpha=0.6, label="normal", density=True)
ax.hist(s[y_test == 1], bins=30, alpha=0.6, label="fraud", density=True)
ax.axvline(THRESHOLD_IF, color="k", ls="--", label="threshold")
ax.set_title("Isolation Forest score (test)")
ax.legend()
plt.tight_layout()
plt.show()


## 7) Evaluation (precision, recall, F1, FPR; ROC; PR; confusion matrices)

Course rubric: ROC and PR curves; confusion matrix at selected threshold; FPR in printed metrics.


In [ ]:
def plot_roc_pr(y_true, scores, title_prefix: str):
    fig, ax = plt.subplots(1, 2, figsize=(10, 4))
    RocCurveDisplay.from_predictions(y_true, scores, name=title_prefix).plot(ax=ax[0])
    PrecisionRecallDisplay.from_predictions(y_true, scores, name=title_prefix).plot(ax=ax[1])
    plt.suptitle(title_prefix)
    plt.tight_layout()
    plt.show()


plot_roc_pr(y_test, scores_stat_test, "Statistical (test)")
plot_roc_pr(y_test, test_result.anomaly_score, "Isolation Forest (test)")

if MEMBER_B_READY and scores_dist_test is not None:
    plot_roc_pr(y_test, scores_dist_test, "Distance-based (test)")
else:
    print("ROC/PR: Member B skipped.")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
ConfusionMatrixDisplay.from_predictions(y_test, pred_stat_test, ax=axes[0])
axes[0].set_title("Statistical — test")
ConfusionMatrixDisplay.from_predictions(y_test, test_result.pred_label, ax=axes[1])
axes[1].set_title("Isolation Forest — test")
plt.tight_layout()
plt.show()

if MEMBER_B_READY and scores_dist_test is not None and thr_dist is not None:
    pred_b = (scores_dist_test >= thr_dist).astype(int)
    fig, ax = plt.subplots(figsize=(4, 4))
    ConfusionMatrixDisplay.from_predictions(y_test, pred_b, ax=ax)
    ax.set_title("Distance-based — test")
    plt.show()
else:
    print("Confusion matrix: Member B skipped.")


## 8) Interpretability — Member C figures (optional)

Embedded if present under `artifacts/`.


In [ ]:
for name in (
    "iforest_confusion_matrix_test.png",
    "iforest_score_hist_test.png",
    "iforest_pca_val_scores.png",
    "iforest_feature_delta_topN.png",
):
    p = ARTIFACT_DIR / name
    if p.is_file():
        print(name)
        idisplay(Image(filename=str(p)))
    else:
        print("Missing (optional):", p)


## 9) Comparative discussion (short)

| Method | Owner | Notes |
|--------|-------|--------|
| Robust-z statistical | Member A slot | Fast, interpretable; threshold from val F1. |
| Distance-based | Member B | Add `memberB_scores.npz` + `MEMBER_B_READY`. |
| Isolation Forest | Member C | Saved model + threshold; stronger feature interactions. |


## 10) Reflection (expand in separate doc if required)

- Threshold trade-offs (precision vs recall vs FPR; fraud vs investigation cost).
- Impact of **class imbalance** on metrics choice.
- Production failure modes and **monitoring** / **retraining** (Member C + team).
- **MLflow:** re-run `memberC_ml_tracking.ipynb` for logged runs / `mlflow_run_summary.json`.


## 11) Reproducibility

- Seed `42` matches `data_pipeline`.
- Dependencies: `requirements.txt`.
- Member C metrics JSON: `artifacts/iforest_metrics_memberC.json` (from Member C notebook).
